# Naive Bayes: SMS Spam Classification
## Beginner-friendly practical notebook

In this notebook, we will train a **Naive Bayes classifier** to identify messages as **Spam** or **Not Spam**.

### Learning objectives
- Understand what text classification means.
- Convert text into numeric features using word counts.
- Train a Multinomial Naive Bayes model.
- Evaluate it using accuracy, precision, recall, F1-score, and a confusion matrix.
- Predict the category of new messages.

> This notebook uses a small educational dataset created for learning. Metrics on this dataset are illustrative and should not be treated as real-world benchmark results.


## 1. Import libraries\nWe use pandas for data handling, scikit-learn for splitting, text vectorization, model training, and evaluation, plus matplotlib for plotting.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

## 2. Load the dataset
The CSV contains two columns:
- `Message`: the text message
- `Category`: its correct label (`Spam` or `Not Spam`)

The model learns patterns from the labeled examples.

In [ ]:
# If the CSV is saved in the same folder as this notebook:
df = pd.read_csv("naive_bayes_sms_dataset.csv")

# View a few records
df.head()

## 3. Explore the data\nFirst, check how many messages belong to each class. This helps us understand whether the dataset is balanced.

In [ ]:
print("Dataset shape:", df.shape)
print("\\nClass counts:")
print(df["Category"].value_counts())

df["Category"].value_counts().plot(kind="bar", title="Number of messages per class")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

## 4. Separate features and target\n- **X (features):** message text that the model will use as clues.\n- **y (target):** the answer the model should learn to predict.

In [ ]:
X = df["Message"]
y = df["Category"]

print("Features example:", X.iloc[0])
print("Target example:", y.iloc[0])

## 5. Train-test split
We keep some messages aside for testing. The model must not learn from test messages during training.

- Training set: examples used to learn.
- Test set: unseen examples used to estimate performance.
- `stratify=y`: keeps the class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

## 6. Convert text to numbers with CountVectorizer
Machine-learning algorithms generally work with numbers, not raw sentences. `CountVectorizer`:
1. Builds a vocabulary from the training messages.
2. Counts how many times each vocabulary word appears in each message.
3. Creates a numeric matrix.

We fit the vectorizer **only on training data** to avoid data leakage.

In [ ]:
vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Training matrix shape:", X_train_vec.shape)
print("Test matrix shape:", X_test_vec.shape)
print("Sample vocabulary:", vectorizer.get_feature_names_out()[:25])

### Peek at the word-count matrix
Rows represent messages and columns represent vocabulary words. Each cell contains a word count.

A value of `1` means that word appears once in that message; `0` means it does not appear.

In [ ]:
# Show a small readable portion of the matrix
word_matrix = pd.DataFrame(
    X_train_vec.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=X_train.index
)
word_matrix.iloc[:5, :12]

## 7. Train Multinomial Naive Bayes
Multinomial Naive Bayes is a common Naive Bayes variant for text represented as counts.

- `fit()` learns class probabilities and word likelihoods from the training messages.
- `alpha=1.0` applies Laplace smoothing by default in this setup.
- Smoothing helps prevent zero probabilities for words not seen in a class.

In [ ]:
model = MultinomialNB(alpha=1.0)
model.fit(X_train_vec, y_train)

print("Model training complete.")

## 8. Predict the test messages\nThe model has not seen these test messages during training. It now predicts their categories.

In [ ]:
y_pred = model.predict(X_test_vec)

results = pd.DataFrame({
    "Message": X_test,
    "Actual": y_test,
    "Predicted": y_pred
})
results.head(10)

## 9. Evaluate model performance
We examine more than accuracy:
- **Accuracy:** proportion of all predictions that are correct.
- **Precision for Spam:** among messages predicted as Spam, how many are truly Spam?
- **Recall for Spam:** among all actual Spam messages, how many did we detect?
- **F1-score:** balances precision and recall.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision_spam = precision_score(y_test, y_pred, pos_label="Spam", zero_division=0)
recall_spam = recall_score(y_test, y_pred, pos_label="Spam", zero_division=0)
f1_spam = f1_score(y_test, y_pred, pos_label="Spam", zero_division=0)

print(f"Accuracy:       {accuracy:.3f}")
print(f"Spam precision: {precision_spam:.3f}")
print(f"Spam recall:    {recall_spam:.3f}")
print(f"Spam F1-score:  {f1_spam:.3f}")

print("\\nFull classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 10. Confusion matrix
The confusion matrix shows how many messages were correctly or incorrectly assigned to each class.

- True positive (Spam correctly predicted Spam)
- True negative (Not Spam correctly predicted Not Spam)
- False positive (Not Spam incorrectly flagged as Spam)
- False negative (Spam missed and predicted Not Spam)

For spam filtering, false positives matter because a legitimate message could be incorrectly sent to the spam folder.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    labels=["Spam", "Not Spam"],
    cmap="Blues",
    values_format="d"
)
plt.title("Naive Bayes — Confusion Matrix")
plt.show()

## 11. Predict new messages
We can pass brand-new text to the model. The vectorizer transforms the words using the vocabulary learned during training, then the model predicts the category.

In [ ]:
new_messages = [
    "Congratulations you won free cash click now",
    "Are we still meeting after class today",
    "Claim your bonus prize before it expires",
    "Please send the project file when you have time"
]

new_vec = vectorizer.transform(new_messages)
new_predictions = model.predict(new_vec)
new_probabilities = model.predict_proba(new_vec)

prediction_table = pd.DataFrame({
    "Message": new_messages,
    "Predicted Category": new_predictions,
    "Spam Probability": new_probabilities[:, list(model.classes_).index("Spam")]
})
prediction_table

## 12. Interpret prediction probabilities
`predict_proba()` returns model-estimated probabilities for each class. These are model estimates based on its learned assumptions; they are not guaranteed to be perfectly calibrated real-world probabilities.

The model's selected class is generally the class with the higher estimated probability.

In [ ]:
# Display class order and probabilities
print("Model classes:", model.classes_)
for msg, probs in zip(new_messages, new_probabilities):
    print("\\nMessage:", msg)
    for label, prob in zip(model.classes_, probs):
        print(f"  {label}: {prob:.3f}")

## 13. What does alpha (Laplace smoothing) change?
If a word never appears in training messages of a particular class, its estimated likelihood could be zero without smoothing. Smoothing adds a small amount to word counts so a single unseen word does not automatically eliminate a class.

Try `alpha=0.1`, `alpha=1.0`, and `alpha=5.0`. Larger smoothing makes the word probabilities less dependent on observed counts.

In [ ]:
for alpha in [0.1, 1.0, 5.0]:
    nb = MultinomialNB(alpha=alpha)
    nb.fit(X_train_vec, y_train)
    p = nb.predict(vectorizer.transform(["free money offer"]))
    print(f"alpha={alpha}: prediction={p[0]}")

## 14. Limitations and responsible use
- This dataset is small and designed for teaching; performance can vary greatly on real messages.
- Naive Bayes treats features as conditionally independent given the class, a simplifying assumption.
- Word counts do not understand full context, sarcasm, or meaning as humans do.
- Real spam systems need larger, diverse data, ongoing evaluation, and attention to false positives/false negatives.

## 15. Student exercises
1. Add 5 more Spam and 5 Not Spam messages. Retrain the model.
2. Compare the model with and without `alpha` smoothing.
3. Try messages that contain the word “free” in a legitimate context.
4. Compare MultinomialNB with BernoulliNB.
5. Explain why the test set should not be used to fit the vectorizer.
6. In a spam-filtering system, which is more costly: a false positive or false negative? Explain your answer.

## Final summary
Naive Bayes uses probability and evidence to predict a class. For text classification, Multinomial Naive Bayes can learn from word counts and provide a quick, interpretable baseline. Always test on data the model did not train on.